In [0]:
import pyspark.sql.functions as F 
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import Row 
from pyspark.sql.functions import col , when

In [0]:
%sql
USE CATALOG liquid_telecom;

### **merging subscription and product tables as product table**

In [0]:
df_subscriptions = spark.table("liquid_telecom.silver.s_dim_subscriptions")
df_product = spark.table("liquid_telecom.silver.s_dim_product")

In [0]:
df_product_enriched = (
    df_product.alias("p")
    .join(
        df_subscriptions.select(
            col("subscription_id"),
            col("subscription_name")
        ).alias("s"),
        on="subscription_id",
        how="left"
    )
)

df_product_enriched = df_product_enriched.filter(col("subscription_name").isNotNull())

In [0]:
desired_columns_order = ["product_id", "product_name", "subscription_id", "subscription_name"]
df_product_enriched = df_product_enriched.select(desired_columns_order)
display(df_product_enriched.limit(10))

product_id,product_name,subscription_id,subscription_name
P000147,Apn Temp Upgrade Increamental,S00000,Generic Solution
P001516,Nyaradzo Kadoma Branch Dia,S00000,Generic Solution
P001660,City Of Harare,S00000,Generic Solution
P001876,Mpls - 1 Mbps-embakasi Branch To Mombasa Maize Miller,S00000,Generic Solution
P005101,Mohamed Badar Al-sinani,S00005,Lma Solution
P007071,Mo Software (heatweb Solution ),S00016,Ms Office365 Solution
P008270,Business Champs,S00023,Africa Data Centres
P009668,2 Seats Business Premium - Kaula Tech,S00094,Microsoft 365 Business Premium
P012938,13 Woodgate,S00018,Ip Vpn
P014276,Wifi Extension,S00010,Misc Equipment


In [0]:
df_product_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("liquid_telecom.gold.g_dim_product")

## **Accounts**

In [0]:
df_account = spark.read.table("liquid_telecom.silver.s_dim_account")

In [0]:
desired_columns_order = ["account_id", "account_number", "account_name"]
df_account = df_account.select(desired_columns_order)
display(df_account.limit(10))

account_id,account_number,account_name
A000000000000,LT-ACC-676509,Ambassade De La Rdc-kigali
A000000000001,LT-ACC-1036221,Test Account - Dataport Sdn Ort Account
A000000000002,LT-ACC-1039803,Cristina Polato
A000000000003,LT-ACC-10730,Fidelity Life Assurance
A000000000004,LT-ACC-573447,Broadband Cameroon Sas
A000000000005,LT-ACC-989174,Shoprite Checkers (pty) Ltd
A000000000006,LT-ACC-00300,Xtranet Communications Ltd
A000000000007,LT-ACC-431047,Wilmar Industries Zambia Limited
A000000000008,LT-ACC-573640,Dsd Sarl
A000000000009,LT-ACC-00884,Lolc Kenya Microfinance


In [0]:
df_account.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("liquid_telecom.gold.g_dim_account")

## **Country**

In [0]:
df_country = spark.read.table("liquid_telecom.silver.s_dim_country")

In [0]:
df_country = df_country.withColumn(
    "region",
    when(
        col("operating_country").rlike(
            "(?i)rwanda|zimbabwe|zambia|kenya|botswana|drc|africa|sahara|zanlink|raha|east africa"
        ),
        "Africa"
    )
    .when(
        col("operating_country").rlike("(?i)uk|united kingdom"),
        "Europe"
    )
    .when(
        col("operating_country").rlike("(?i)satellite|intelligent technologies"),
        "Global"
    )
    .otherwise("Other")
)

In [0]:
desired_columns = ["country_id", "operating_country", "region"]
df_country = df_country.select(desired_columns)
display(df_country.limit(10))

country_id,operating_country,region
C000,LTR - Liquid Telecommunications Rwanda Ltd,Africa
C001,ZANL - Zanlink,Africa
C002,LTZ - Liquid Telecom Zimbabwe,Africa
C003,LTSAT - Liquid Telecommunications Satellite Services,Global
C004,LTK - Liquid Telecommunications Kenya Ltd,Africa
C005,LTZM - Liquid Telecommunications Zambia Ltd,Africa
C006,INFOCOM - Infocom 2013 Ltd,Other
C007,LTUK - Liquid Telecommunications Ltd,Europe
C008,LTSS - Liquid Telecom Sahara Holdings Ltd,Africa
C009,LTOPSDRC - Liquid Telecommunications Operations DRC S.P.R.L,Africa


In [0]:
df_country.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("liquid_telecom.gold.g_dim_country")

## **Date**

In [0]:
df_date = spark.read.table("liquid_telecom.silver.s_dim_date")

In [0]:
desired_columns = ["date_key", "date", "year", "month_name", "quarter"]
df_date = df_date.select(desired_columns)
display(df_date.limit(10))

date_key,date,year,month_name,quarter
20240731,2024-07-31,2024,July,3
20240802,2024-08-02,2024,August,3
20240824,2024-08-24,2024,August,3
20240911,2024-09-11,2024,September,3
20240913,2024-09-13,2024,September,3
20241013,2024-10-13,2024,October,4
20240604,2024-06-04,2024,June,2
20240612,2024-06-12,2024,June,2
20240704,2024-07-04,2024,July,3
20240705,2024-07-05,2024,July,3


In [0]:
df_date.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("liquid_telecom.gold.g_dim_date")